In [1]:
from fastai.text.all import *

In [2]:
path = untar_data(URLs.HUMAN_NUMBERS)

In [3]:
path.ls()

(#2) [Path('/home/cgb2/.fastai/data/human_numbers/train.txt'),Path('/home/cgb2/.fastai/data/human_numbers/valid.txt')]

In [4]:
lines = L()
with open(path/'train.txt') as f: lines += L(*f.readlines())
with open(path/'valid.txt') as f: lines += L(*f.readlines())

In [5]:
lines

(#9998) ['one \n','two \n','three \n','four \n','five \n','six \n','seven \n','eight \n','nine \n','ten \n'...]

In [151]:
text= 'h e l l o '*100
text

'h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o

In [164]:
tokens= text.split(' ')[:-1]

In [165]:
vocab = L(*tokens).unique()
vocab

(#4) ['h','e','l','o']

In [166]:
word2idx = {w:i for i,w in enumerate(vocab)}
word2idx

{'h': 0, 'e': 1, 'l': 2, 'o': 3}

In [167]:
nums= L(word2idx[i] for i in tokens)

In [168]:
nums

(#500) [0,1,2,2,3,0,1,2,2,3...]

In [169]:
L((tokens[i],tokens[i+1]) for i in range(0,4))

(#4) [('h', 'e'),('e', 'l'),('l', 'l'),('l', 'o')]

In [122]:
L((tokens[i:i+2],tokens[i+2]) for i in range(0,3))

(#3) [(['h', 'e'], 'l'),(['e', 'l'], 'l'),(['l', 'l'], 'o')]

- 이전 세단어를 기반으로 다음단어를 예측하자. 

In [125]:
seqs = L((tensor(nums[i:i+1]), nums[i+1]) for i in range(0,4))
seqs

(#4) [(tensor([0]), 1),(tensor([1]), 2),(tensor([2]), 2),(tensor([2]), 3)]

In [126]:
#bs = 64 
cut = int(len(seqs)*0.8)
dls = DataLoaders.from_dsets(seqs[:cut],seqs[cut:],bs=1,shuffle=False)

In [127]:
dls

In [129]:
class LMModel1(Module): 
    def __init__(self, vocab_sz, n_hidden):
        self.i_h = nn.Embedding(vocab_sz,n_hidden)
        self.h_h = nn.Linear(n_hidden,n_hidden)
        self.h_o = nn.Linear(n_hidden,vocab_sz) 
    def forward(self,x):
        h= F.relu(self.h_h(self.i_h(x[:,0])))
        h= h+ self.i_h(x[:,1])
        h= F.relu(self.h_h(h))
        h= h+self.i_h(x[:,2])
        h= F.relu(self.h_h(h))
        return self.h_o(h)

In [130]:
lrnr = Learner(dls, LMModel1(len(vocab),64), loss_func=F.cross_entropy, metrics=accuracy)

In [131]:
lrnr.fit_one_cycle(1,1e-3)

epoch,train_loss,valid_loss,accuracy,time


IndexError: index 1 is out of bounds for dimension 1 with size 1

In [64]:
x,y = dls.one_batch()

In [65]:
x

tensor([[ 0,  1,  2],
        [ 1,  3,  1],
        [ 4,  1,  5],
        [ 1,  6,  1],
        [ 7,  1,  8],
        [ 1,  9,  1],
        [10,  1, 11],
        [ 1, 12,  1],
        [13,  1, 14],
        [ 1, 15,  1],
        [16,  1, 17],
        [ 1, 18,  1],
        [19,  1, 20],
        [ 1, 20,  0],
        [ 1, 20,  2],
        [ 1, 20,  3],
        [ 1, 20,  4],
        [ 1, 20,  5],
        [ 1, 20,  6],
        [ 1, 20,  7],
        [ 1, 20,  8],
        [ 1, 20,  9],
        [ 1, 21,  1],
        [21,  0,  1],
        [21,  2,  1],
        [21,  3,  1],
        [21,  4,  1],
        [21,  5,  1],
        [21,  6,  1],
        [21,  7,  1],
        [21,  8,  1],
        [21,  9,  1],
        [22,  1, 22],
        [ 0,  1, 22],
        [ 2,  1, 22],
        [ 3,  1, 22],
        [ 4,  1, 22],
        [ 5,  1, 22],
        [ 6,  1, 22],
        [ 7,  1, 22],
        [ 8,  1, 22],
        [ 9,  1, 23],
        [ 1, 23,  0],
        [ 1, 23,  2],
        [ 1, 23,  3],
        [ 

In [71]:
y

tensor([ 1,  4,  1,  7,  1, 10,  1, 13,  1, 16,  1, 19,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1, 21, 21, 21, 21, 21, 21, 21, 21, 21, 22,  0,  2,  3,  4,
         5,  6,  7,  8,  9,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, 24, 24, 24,
        24, 24, 24, 24, 24, 24, 25,  0,  2,  3])

In [105]:
lrnr.model(x).detach()[1,:]

tensor([ 0.2288, -3.7252,  0.2827,  0.2769, -0.1205,  0.7285,  0.1120,  0.4477,
        -0.2614, -1.4712, -3.5619, -3.8187, -4.4012, -3.4710, -3.0671, -4.0535,
        -3.7887, -4.0696, -4.0388, -3.5772, -2.4696, -1.9871, -2.1546, -3.0250,
        -1.7772, -2.3107, -3.1036, -1.3761, -2.6543, -2.4458])

In [91]:
x[50]

tensor([ 1, 23,  9])

In [97]:
lrnr.predict(x)[0]

tensor([-3.0642,  0.3804, -3.1812, -3.6344, -2.6501, -3.6768, -3.3363, -3.1496,
        -3.4109, -3.3065, -4.9465, -4.9160, -5.0131, -4.6013, -4.5047, -4.0049,
        -4.3536, -4.8198, -4.4614, -4.7675, -3.4891, -3.1594, -3.3581, -2.8839,
        -4.0839, -2.7119, -3.3432, -3.1585,  2.1528,  5.3299])

In [88]:
x

tensor([[ 0,  1,  2],
        [ 1,  3,  1],
        [ 4,  1,  5],
        [ 1,  6,  1],
        [ 7,  1,  8],
        [ 1,  9,  1],
        [10,  1, 11],
        [ 1, 12,  1],
        [13,  1, 14],
        [ 1, 15,  1],
        [16,  1, 17],
        [ 1, 18,  1],
        [19,  1, 20],
        [ 1, 20,  0],
        [ 1, 20,  2],
        [ 1, 20,  3],
        [ 1, 20,  4],
        [ 1, 20,  5],
        [ 1, 20,  6],
        [ 1, 20,  7],
        [ 1, 20,  8],
        [ 1, 20,  9],
        [ 1, 21,  1],
        [21,  0,  1],
        [21,  2,  1],
        [21,  3,  1],
        [21,  4,  1],
        [21,  5,  1],
        [21,  6,  1],
        [21,  7,  1],
        [21,  8,  1],
        [21,  9,  1],
        [22,  1, 22],
        [ 0,  1, 22],
        [ 2,  1, 22],
        [ 3,  1, 22],
        [ 4,  1, 22],
        [ 5,  1, 22],
        [ 6,  1, 22],
        [ 7,  1, 22],
        [ 8,  1, 22],
        [ 9,  1, 23],
        [ 1, 23,  0],
        [ 1, 23,  2],
        [ 1, 23,  3],
        [ 